In [ ]:
import sys
sys.path.append(".")

from src.utils.models import Qwen3, Qwen35, LFM2, IBM_Granite1b, IBM_Granite, GPT2
from data.preprocessing import c4_dataset
from src.test import test_model
from src.utils.visuals import generate_plots

import json

# Choose dataset configuration
LANGUAGE = "en"  # e.g. "en"
SPLIT = "train"  # e.g. "train"

# Instantiate model wrappers
models = {
    "Qwen3": Qwen3(),
    "Qwen3.5": Qwen35(),
    "LFM2": LFM2(),
    "IBM-G1B": IBM_Granite1b(),
    "IBM-G350M": IBM_Granite(),
    "GPT2": GPT2(),
}

results_files = []

for name, wrapper in models.items():
    print(f"Running metrics for {name}...")
    # Each wrapper has .tokenizer and .model
    dataset = c4_dataset(
        split=SPLIT,
        language=LANGUAGE,
        tokenizer=wrapper.tokenizer,
    )

    metrics = test_model(wrapper.model, dataset)

    json_name = f"{name}_metrics.json"
    with open(json_name, "w") as f:
        # Convert tuple keys to string form "(read,gen)" to match visuals.generate_plots
        serializable = {}
        for (read_len, gen_len), vals in metrics.items():
            key = f"({read_len},{gen_len})"
            serializable[key] = vals
        json.dump(serializable, f)

    results_files.append(json_name)
    print(f"Saved metrics to {json_name}\n")

# Generate comparison plots
if results_files:
    print("Generating plots...")
    generate_plots(results_files, "plots")
    print("Plots saved as plots.png")
